In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor, SpanExporter, SpanExportResult

import pandas as pd
import sqlite3

In [8]:
from starter import rag
from rag_helper import RAGBase


In [9]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = rag.rag(query)
print(answer)

It keeps calling the model in a `while True` loop, and after each response it checks whether the model returned any `function_call` items.

- If there **is** a function call, the code runs the tool, appends the tool output to `messages`, and loops again.
- If there are **no** function calls, it breaks out of the loop and stops.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In short: **the loop repeats until the model returns a response with no tool calls.**


In [10]:
provider = TracerProvider()
# provider.add_span_processor(
#     SimpleSpanProcessor(
#         ConsoleSpanExporter()
#     )
# )

In [11]:
class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [12]:
provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

In [13]:
trace.set_tracer_provider(provider)

In [14]:
tracer = trace.get_tracer("llm-zoomcamp")

# Question 1

In [15]:
class RAGTraced(RAGBase):
    """
    Wraps each pipeline stage in its own span. Spans nest based on
    call order, not on where you *define* them — search() and llm()
    become children of rag() because rag()'s span is still open
    (i.e. "current") when they execute.
    """

    def search(self, query: str, num_results: int = 5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("search.query", query)
            span.set_attribute("search.num_results_requested", num_results)
            results = super().search(query, num_results=num_results)
            span.set_attribute("search.num_results_returned", len(results))
            return results

    def llm(self, prompt: str):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("llm.model", self.model)
            span.set_attribute("llm.prompt_chars", len(prompt))
            # The call has to happen before we can read anything off
            # its result — usage doesn't exist until the response does.
            response = super().llm(prompt)

            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)

            return response

    def rag(self, query: str) -> str:
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("rag.query", query)
            return super().rag(query)

In [16]:
# rt = RAGTraced(index = rag.index, llm_client = rag.llm_client)
# answer = rt.rag("How does the agentic loop keep calling the model until it stops?")

In [17]:
answer_question_1 = 3

# Question 2

In [18]:
answer_question_2 = 7000

# Question 3

In [19]:
answer_question_3 = "Slighty less than 2 seconds and the question seems to imply I shouldn't pick the highest value."

# Question 4

In [24]:
rt = RAGTraced(index = rag.index, llm_client = rag.llm_client)
answer = rt.rag("How does the agentic loop keep calling the model until it stops?")

question_4_answer = ["search", "llm", "rag"]

# Question 5

In [ ]:
def load_span_durations(db_path: str = "traces.db") -> pd.DataFrame:
    """
    Reads spans from the SQLite trace store and computes per-span
    duration in milliseconds. Returns one row per span, not yet
    aggregated — aggregation happens separately so you can inspect
    both the raw distribution and the rollup.
    """
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(
            "SELECT name, start_time, end_time FROM spans", conn
        )

    # start_time/end_time come out of OTel's console/SQLite exporters as
    # ISO 8601 strings with a 'Z' suffix (UTC). pd.to_datetime handles this,
    # but only if you don't skip this step — comparing two strings like
    # "2026-07-15T20:32:57.421Z" > "2026-07-15T20:32:55.294Z" *happens* to
    # work lexicographically for ISO timestamps, but relying on that is
    # fragile (breaks the moment timezone offsets or non-padded fields
    # show up) — always parse to datetime explicitly before subtracting.
    df["start_time"] = pd.to_datetime(df["start_time"])
    df["end_time"] = pd.to_datetime(df["end_time"])

    df["duration_ms"] = (
        (df["end_time"] - df["start_time"]).dt.total_seconds() * 1000
    )
    return df


def summarize_by_span_name(df: pd.DataFrame) -> pd.DataFrame:
    """
    Excludes the 'rag' root span (its duration double-counts everything
    beneath it, since it wraps both children) and aggregates the rest.
    """
    children = df[df["name"] != "rag"]

    summary = (
        children.groupby("name")["duration_ms"]
        .agg(total_ms="sum", mean_ms="mean", count="count")
        .sort_values("total_ms", ascending=False)
    )
    return summary


df = load_span_durations("traces.db")
summary = summarize_by_span_name(df)
print(summary)

           total_ms      mean_ms  count
name                                   
llm     4426.934632  2213.467316      2
search     7.570191     3.785095      2


In [ ]:
answer_question_5 = "llm"

# Question 6

In [ ]:
answer_question_6 = "7111 every single time"